# Visualization: Depots and Stores Map

Notebook này đọc dữ liệu `depots.csv` và `stores_hcm_real.csv` từ thư mục `02_processed` và hiển thị trên bản đồ tương tác sử dụng Folium.

In [1]:
# 1. Setup và Import thư viện
import pandas as pd
import folium
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Đọc dữ liệu
# Đường dẫn dựa trên cấu trúc cây thư mục bạn cung cấp
base_path = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/02_processed/'

try:
    df_depots = pd.read_csv(f'{base_path}depots.csv')
    df_stores = pd.read_csv(f'{base_path}stores_hcm_real.csv')

    print("--- Depots Data ---")
    display(df_depots.head())
    print("\n--- Stores Data ---")
    display(df_stores.head())
except FileNotFoundError:
    print("Không tìm thấy file. Vui lòng kiểm tra lại đường dẫn base_path.")

--- Depots Data ---


,depot_id,name,address,lat,long
0,DEPOT_001,Saigon Express Quận 7,"34 Đào Trí, Phú Thuận, Q.7",10.735,106.73
1,DEPOT_002,TBS Logistics Dĩ An,"Số 9 ĐT743, Bình Thắng, Dĩ An",10.890,106.82
2,DEPOT_003,SGV Logistics Tân Bình,"Đường CN1, KCN Tân Bình, Tân Phú",10.810,106.62



--- Stores Data ---


,store_id_mapped,type,name,address,lat,long,note
0,STORE0001,Hypermarket,Emart Gò Vấp,"366 Phan Văn Trị, P.5, Gò Vấp",10.8231,106.6935,Hub phía Bắc
1,STORE0004,Hypermarket,Lotte Mart Quận 7,"469 Nguyễn Hữu Thọ, Tân Hưng, Q.7",10.7409,106.7018,Hub phía Nam
2,STORE0005,Hypermarket,Aeon Mall Tân Phú,"30 Bờ Bao Tân Thắng, Sơn Kỳ, Tân Phú",10.8016,106.6174,Hub phía Tây
3,STORE0006,Hypermarket,Gigamall Thủ Đức,"242 Phạm Văn Đồng, Hiệp Bình Chánh, Thủ Đức",10.8277,106.7216,Hub phía Đông
4,STORE0008,Hypermarket,MM Mega Market An Phú,"Khu B, KĐT An Phú An Khánh, Thủ Đức",10.8002,106.7432,Hub phía Đông (Metro cũ)


In [6]:
# 3. Khởi tạo bản đồ và Vẽ tọa độ

# Lấy tọa độ trung tâm (lấy trung bình của các kho hoặc mặc định TP.HCM)
center_lat = 10.762622
center_long = 106.660172

# Tạo map base
m = folium.Map(location=[center_lat, center_long], zoom_start=11, tiles='OpenStreetMap')

# --- Vẽ DEPOTS (Kho) ---
# Icon: warehouse, Màu: Đỏ (Red)
for idx, row in df_depots.iterrows():
    popup_content = f"<b>{row['name']}</b><br>ID: {row['depot_id']}<br>{row['address']}"

    folium.Marker(
        location=[row['lat'], row['long']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=f"Depot: {row['name']}",
        icon=folium.Icon(color='red', icon='warehouse', prefix='fa')
    ).add_to(m)

# --- Vẽ STORES (Siêu thị) theo Type ---
# Định nghĩa màu sắc cho từng loại
type_color_map = {
    'Supermarket': 'blue',      # Siêu thị: Xanh dương
    'Hypermarket': 'darkblue',  # Đại siêu thị: Xanh đậm
    'Convenience': 'green',     # Tiện lợi: Xanh lá
    'E-commerce': 'purple'      # TMĐT: Tím
}

for idx, row in df_stores.iterrows():
    store_type = row['type']

    # Lấy màu dựa trên dictionary, nếu type lạ thì để màu cam (orange)
    icon_color = type_color_map.get(store_type, 'orange')

    popup_content = f"<b>{row['name']}</b><br>ID: {row['store_id_mapped']}<br>Type: {store_type}"

    folium.Marker(
        location=[row['lat'], row['long']],
        popup=folium.Popup(popup_content, max_width=300),
        tooltip=f"Store: {row['name']}",
        icon=folium.Icon(color=icon_color, icon='shopping-cart', prefix='fa')
    ).add_to(m)

# --- THÊM MỤC LỤC (LEGEND) VÀO GÓC TRÁI ---
from folium import Element

legend_html = '''
 <div style="
     position: fixed;
     bottom: 50px; left: 50px; width: 170px; height: auto;
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white;
     opacity: 0.9;
     padding: 10px;
     border-radius: 5px;
     font-family: sans-serif;
     box-shadow: 3px 3px 5px rgba(0,0,0,0.3);
 ">
     <b>Chú thích (Legend)</b><br>
     <i class="fa fa-warehouse" style="color:red; margin-top:5px"></i>&nbsp; Kho (Depot)<br>
     <i class="fa fa-shopping-cart" style="color:blue; margin-top:5px"></i>&nbsp; Supermarket<br>
     <i class="fa fa-shopping-cart" style="color:green; margin-top:5px"></i>&nbsp; Convenience<br>
     <i class="fa fa-shopping-cart" style="color:purple; margin-top:5px"></i>&nbsp; E-commerce<br>
     <i class="fa fa-shopping-cart" style="color:orange; margin-top:5px"></i>&nbsp; Khác<br>
 </div>
 '''

m.get_root().html.add_child(Element(legend_html))

# Hiển thị bản đồ
m

In [7]:
# 4. (Tùy chọn) Lưu bản đồ thành file HTML vào folder output
output_path = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/03_output/visualization_map.html'
m.save(output_path)
print(f"Đã lưu bản đồ tại: {output_path}")

Đã lưu bản đồ tại: /content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/03_output/visualization_map.html
